# Emotion-Aware Voice Cloning - Dataset Preparation and Training

This notebook guides you through:
1. Downloading required datasets (VCTK, EmoV-DB, LibriTTS)
2. Preprocessing audio files
3. Preparing training data
4. Training the voice cloning model
5. Generating voice samples

## Requirements
- Kaggle GPU (P100)
- Internet access enabled
- At least 50GB disk space

In [ ]:
# Install required packages
!pip install --upgrade pip
!pip install torch==2.1.0 torchaudio==2.1.0
!pip install TTS==0.17.6
!pip install librosa==0.10.1
!pip install soundfile==0.12.1
!pip install wandb==0.15.12
!pip install tqdm
!pip install pyyaml
!pip install pandas
!pip install matplotlib
!pip install IPython
!pip install kaggle

# Import required libraries
import os
import torch
import torchaudio
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
import yaml
import matplotlib.pyplot as plt
from IPython.display import Audio
import warnings
warnings.filterwarnings('ignore')

# Dataset Download and Setup

We'll download and prepare three main datasets:
1. **VCTK**: Multi-speaker dataset with clean recordings
2. **EmoV-DB**: Emotional speech dataset
3. **LibriTTS**: Large-scale audiobook dataset

First, set up your Kaggle API credentials to download datasets.

In [ ]:
# Create directory structure
base_dir = Path('/kaggle/working/voice-clone-tool')
dirs = [
    base_dir / 'data/raw_audio/vctk',
    base_dir / 'data/raw_audio/emov_db',
    base_dir / 'data/raw_audio/libritts',
    base_dir / 'data/processed',
    base_dir / 'checkpoints',
    base_dir / 'logs'
]

for dir_path in dirs:
    dir_path.mkdir(parents=True, exist_ok=True)

# Download datasets
def download_vctk():
    """Download VCTK dataset"""
    print("Downloading VCTK dataset...")
    !wget -q --show-progress https://datashare.ed.ac.uk/bitstream/handle/10283/3443/VCTK-Corpus-0.92.zip
    !unzip -q VCTK-Corpus-0.92.zip -d data/raw_audio/vctk
    !rm VCTK-Corpus-0.92.zip
    print("VCTK dataset downloaded and extracted")

def download_emov_db():
    """Download EmoV-DB dataset"""
    print("Downloading EmoV-DB dataset...")
    !kaggle datasets download -d pasyutinna/emov-db
    !unzip -q emov-db.zip -d data/raw_audio/emov_db
    !rm emov-db.zip
    print("EmoV-DB dataset downloaded and extracted")

def download_libritts():
    """Download LibriTTS dataset (dev-clean subset)"""
    print("Downloading LibriTTS dataset...")
    !wget -q --show-progress https://www.openslr.org/resources/60/dev-clean.tar.gz
    !tar -xzf dev-clean.tar.gz -C data/raw_audio/libritts
    !rm dev-clean.tar.gz
    print("LibriTTS dataset downloaded and extracted")

# Download all datasets
try:
    download_vctk()
    download_emov_db()
    download_libritts()
    print("All datasets downloaded successfully!")
except Exception as e:
    print(f"Error downloading datasets: {e}")

# Verify downloads
def count_audio_files(directory):
    """Count audio files in directory"""
    audio_files = list(Path(directory).rglob("*.wav"))
    return len(audio_files)

print("\nDataset Statistics:")
print(f"VCTK files: {count_audio_files(base_dir/'data/raw_audio/vctk')}")
print(f"EmoV-DB files: {count_audio_files(base_dir/'data/raw_audio/emov_db')}")
print(f"LibriTTS files: {count_audio_files(base_dir/'data/raw_audio/libritts')}")

# Audio Preprocessing

Now we'll preprocess the audio files to ensure consistent format and quality:
1. Convert to consistent sample rate (22050 Hz)
2. Convert to mono
3. Remove silence
4. Normalize audio
5. Create emotion annotations

In [ ]:
# Audio preprocessing functions
def load_and_preprocess_audio(file_path, target_sr=22050):
    """Load and preprocess audio file"""
    try:
        # Load audio
        waveform, sr = librosa.load(file_path, sr=None)
        
        # Resample if necessary
        if sr != target_sr:
            waveform = librosa.resample(waveform, orig_sr=sr, target_sr=target_sr)
        
        # Convert to mono if necessary
        if len(waveform.shape) > 1:
            waveform = librosa.to_mono(waveform)
        
        # Remove silence
        waveform, _ = librosa.effects.trim(waveform, top_db=30)
        
        # Normalize
        waveform = librosa.util.normalize(waveform)
        
        return waveform, target_sr
    
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None, None

def create_emotion_annotations():
    """Create emotion annotations for the datasets"""
    emotions = {
        'neutral': 0,
        'happy': 1,
        'sad': 2,
        'angry': 3,
        'fear': 4
    }
    
    annotations = []
    
    # EmoV-DB annotations (already has emotions)
    emov_dir = base_dir/'data/raw_audio/emov_db'
    for emotion_dir in emov_dir.glob("*"):
        if emotion_dir.is_dir():
            emotion = emotion_dir.name.lower()
            if emotion in emotions:
                for audio_file in emotion_dir.glob("*.wav"):
                    annotations.append({
                        'file': str(audio_file),
                        'emotion': emotion,
                        'dataset': 'emov_db'
                    })
    
    # VCTK (mostly neutral)
    vctk_dir = base_dir/'data/raw_audio/vctk'
    for audio_file in vctk_dir.rglob("*.wav"):
        annotations.append({
            'file': str(audio_file),
            'emotion': 'neutral',
            'dataset': 'vctk'
        })
    
    # LibriTTS (mix of emotions based on text analysis - simplified)
    libritts_dir = base_dir/'data/raw_audio/libritts'
    for audio_file in libritts_dir.rglob("*.wav"):
        # Simple random assignment for demonstration
        # In practice, you'd want to analyze the text for emotion
        emotion = np.random.choice(['neutral', 'happy', 'sad'], p=[0.6, 0.2, 0.2])
        annotations.append({
            'file': str(audio_file),
            'emotion': emotion,
            'dataset': 'libritts'
        })
    
    # Save annotations
    df = pd.DataFrame(annotations)
    df.to_csv(base_dir/'data/emotions.csv', index=False)
    print(f"Created annotations for {len(df)} files")
    return df

# Create annotations
emotions_df = create_emotion_annotations()

# Display statistics
print("\nEmotion Distribution:")
print(emotions_df['emotion'].value_counts())

# Prepare Training Data

Now we'll:
1. Process all audio files
2. Extract mel spectrograms and other features
3. Split into train/val/test sets
4. Create data loaders

In [ ]:
# Process all files and prepare features
def process_file_to_features(row, target_sr=22050):
    """Process a single file and extract features"""
    try:
        # Load and preprocess audio
        waveform, sr = load_and_preprocess_audio(row['file'], target_sr)
        if waveform is None:
            return None
            
        # Calculate mel spectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=waveform,
            sr=sr,
            n_fft=1024,
            hop_length=256,
            win_length=1024,
            n_mels=80
        )
        mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Extract F0 (fundamental frequency)
        f0 = librosa.yin(
            waveform,
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=sr,
            hop_length=256
        )
        
        # Calculate energy
        energy = librosa.feature.rms(y=waveform, hop_length=256)[0]
        
        # Get duration
        duration = len(waveform) / sr
        
        # Save features
        save_dir = base_dir/'data/processed'/row['dataset']
        save_dir.mkdir(parents=True, exist_ok=True)
        
        filename = Path(row['file']).stem
        np.savez(
            save_dir/f"{filename}_features.npz",
            mel_spectrogram=mel_spec,
            f0=f0,
            energy=energy,
            duration=duration,
            emotion=row['emotion']
        )
        
        return True
        
    except Exception as e:
        print(f"Error processing {row['file']}: {e}")
        return None

# Process all files
print("Processing files and extracting features...")
results = []
for idx, row in tqdm(emotions_df.iterrows(), total=len(emotions_df)):
    result = process_file_to_features(row)
    results.append(result)

successful = sum(1 for r in results if r is not None)
print(f"\nSuccessfully processed {successful} out of {len(results)} files")

# Create train/val/test splits
def create_data_splits():
    """Create dataset splits"""
    processed_dir = base_dir/'data/processed'
    all_features = []
    
    for dataset_dir in ['vctk', 'emov_db', 'libritts']:
        features = list((processed_dir/dataset_dir).glob("*_features.npz"))
        all_features.extend(features)
    
    # Shuffle
    np.random.shuffle(all_features)
    
    # Split
    train_ratio, val_ratio = 0.8, 0.1
    train_idx = int(len(all_features) * train_ratio)
    val_idx = int(len(all_features) * (train_ratio + val_ratio))
    
    splits = {
        'train': all_features[:train_idx],
        'val': all_features[train_idx:val_idx],
        'test': all_features[val_idx:]
    }
    
    # Save splits
    for split, files in splits.items():
        with open(base_dir/f'data/{split}_files.txt', 'w') as f:
            for file in files:
                f.write(f"{file}\n")
    
    return splits

# Create splits
splits = create_data_splits()
for split, files in splits.items():
    print(f"{split} set: {len(files)} files")

# Model Training

Now that we have our data prepared, let's:
1. Initialize the voice cloning model
2. Set up training configuration
3. Train the model
4. Track progress with Weights & Biases (optional)

In [ ]:
# Validate setup before training
def validate_setup():
    """Validate data and model setup before training"""
    print("Validating setup...")
    
    # Check processed data
    processed_files = list(Path(base_dir/'data/processed').rglob("*_features.npz"))
    if not processed_files:
        raise ValueError("No processed feature files found!")
    print(f"Found {len(processed_files)} processed files")
    
    # Load a sample file to verify features
    sample_data = np.load(processed_files[0])
    required_keys = ['mel_spectrogram', 'f0', 'energy', 'duration', 'emotion']
    missing_keys = [key for key in required_keys if key not in sample_data]
    if missing_keys:
        raise ValueError(f"Missing features in processed files: {missing_keys}")
    print("Feature format validated")
    
    # Verify model configuration
    required_config = ['model', 'training', 'data', 'checkpoint', 'logging']
    missing_config = [key for key in required_config if key not in config]
    if missing_config:
        raise ValueError(f"Missing configuration sections: {missing_config}")
    print("Configuration validated")
    
    # Test model initialization
    try:
        model = UltimateVoiceClone(training_mode=True, config=config)
        print("Model initialization successful")
        
        # Test batch processing
        sample_batch = next(iter(train_loader))
        output = model.train_step(sample_batch)
        print("Batch processing successful")
        print(f"Initial loss: {output['loss']:.4f}")
        
    except Exception as e:
        raise RuntimeError(f"Model validation failed: {e}")
    
    print("\nAll validations passed! Ready to start training.")

# Run validation
validate_setup()

In [ ]:
# Clone repository and get model code
!git clone https://github.com/Javid-Shaik/voice-cloning-tool.git
%cd voice-cloning-tool

# Import our model
from src.voice_clone import UltimateVoiceClone
from src.utils.logging import setup_logger

# Initialize wandb (optional)
import wandb
wandb.login()  # You'll need to enter your API key

# Load configuration
with open('configs/training.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize model
model = UltimateVoiceClone(training_mode=True, config=config)

# Create data loaders
from torch.utils.data import DataLoader
from src.dataset import VoiceCloneDataset

# Create datasets
train_dataset = VoiceCloneDataset(
    data_dir=base_dir/'data/processed',
    config=config,
    split="train"
)

val_dataset = VoiceCloneDataset(
    data_dir=base_dir/'data/processed',
    config=config,
    split="val"
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    num_workers=config['data']['num_workers'],
    pin_memory=True,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['evaluation']['batch_size'],
    num_workers=config['data']['num_workers'],
    pin_memory=True
)

# Training loop
num_epochs = config['training']['num_epochs']
logger = setup_logger('train', config['logging']['log_dir'])

for epoch in range(num_epochs):
    model.train()
    train_losses = []
    
    # Training
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        metrics = model.train_step(batch)
        train_losses.append(metrics['loss'])
        
        # Log to wandb
        if wandb.run is not None:
            wandb.log({
                'epoch': epoch,
                'train_loss': metrics['loss'],
                'reconstruction_loss': metrics['reconstruction_loss'],
                'emotion_loss': metrics['emotion_loss'],
                'prosody_loss': metrics['prosody_loss']
            })
    
    # Validation
    model.eval()
    val_losses = []
    
    for batch in val_loader:
        metrics = model.validate_step(batch)
        val_losses.append(metrics['loss'])
    
    # Log epoch metrics
    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses)
    
    logger.info(f"Epoch {epoch+1}/{num_epochs}")
    logger.info(f"Train Loss: {avg_train_loss:.4f}")
    logger.info(f"Val Loss: {avg_val_loss:.4f}")
    
    # Save checkpoint
    if (epoch + 1) % config['checkpoint']['save_frequency'] == 0:
        model.save_checkpoint(
            base_dir/f'checkpoints/checkpoint_epoch_{epoch+1}.pt'
        )

print("Training completed!")

# Save final model
model.save_checkpoint(base_dir/'checkpoints/final_model.pt')

# Generate Voice Samples

Let's test our trained model by generating some voice samples with different emotions.

In [ ]:
# Load trained model
model = UltimateVoiceClone()
model.load_checkpoint(base_dir/'checkpoints/final_model.pt')

# Test voice sample (use the first file from test set)
test_file = splits['test'][0]
test_text = "This is a test of emotion-aware voice cloning. I hope you enjoy it!"

# Generate samples with different emotions
emotions = ['happy', 'sad', 'excited', 'angry', 'neutral']
audio_samples = {}

for emotion in emotions:
    output_file = base_dir/f'data/output/test_{emotion}.wav'
    output_file.parent.mkdir(exist_ok=True)
    
    try:
        path = model.generate_ultra_fast(
            text=test_text,
            voice_sample=str(test_file),
            output_file=str(output_file),
            emotion=emotion,
            quality_mode='balanced'
        )
        audio_samples[emotion] = path
        print(f"Generated {emotion} sample: {path}")
    except Exception as e:
        print(f"Error generating {emotion} sample: {e}")

# Play samples
for emotion, path in audio_samples.items():
    print(f"\n{emotion.title()} sample:")
    display(Audio(path))

# Emotion-Aware Voice Cloning System

This notebook sets up and runs the voice cloning system on Kaggle.

## Setup Steps:
1. Install required dependencies
2. Clone project repository
3. Set up data directories
4. Prepare training data
5. Train the model

In [ ]:
# Install dependencies
!pip install torch==2.1.0 torchaudio==2.1.0
!pip install TTS==0.17.6
!pip install librosa==0.10.1
!pip install soundfile==0.12.1
!pip install wandb==0.15.12
!pip install tqdm
!pip install pyyaml

In [ ]:
# Clone the repository
!git clone https://github.com/Javid-Shaik/voice-cloning-tool.git
%cd voice-cloning-tool

In [ ]:
# Create necessary directories
!mkdir -p data/raw_audio
!mkdir -p data/processed
!mkdir -p checkpoints
!mkdir -p logs

## Data Preparation

Upload your voice samples to the `/kaggle/working/voice-cloning-tool/data/raw_audio` directory. 
If you have emotion annotations, create a CSV file with format: `filename,emotion,intensity`

In [ ]:
# Optional: Download sample dataset (VCTK subset)
!kaggle datasets download -d toponymo/vctk-emotions-subset
!unzip vctk-emotions-subset.zip -d data/raw_audio

In [ ]:
# Prepare training data
!python prepare_data.py \
    --input-dir data/raw_audio \
    --output-dir data/processed \
    --emotions-file data/raw_audio/emotions.csv \
    --config configs/training.yaml

## Model Training

Set up Weights & Biases for experiment tracking (optional but recommended)

In [ ]:
# Initialize wandb (optional)
import wandb
wandb.login()

# Train the model
!python train.py --config configs/training.yaml

## Testing the Model

Test the trained model with a sample voice

In [ ]:
from src.voice_clone import UltimateVoiceClone
import yaml

# Load config
with open('configs/training.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Initialize model
cloner = UltimateVoiceClone()

# Load trained checkpoint
cloner.load_checkpoint('checkpoints/best_model.pt')

# Test voice cloning
test_text = "Hello, this is a test of emotion-aware voice cloning."
voice_sample = "data/raw_audio/sample.wav"  # Replace with your sample
output_file = "data/output/test_output.wav"

# Generate speech with different emotions
emotions = ['happy', 'sad', 'excited', 'neutral']

for emotion in emotions:
    output = cloner.generate_ultra_fast(
        text=test_text,
        voice_sample=voice_sample,
        output_file=f"data/output/test_{emotion}.wav",
        emotion=emotion,
        quality_mode='balanced'
    )
    print(f"Generated {emotion} sample: {output}")

# Play a sample
from IPython.display import Audio
Audio(filename="data/output/test_happy.wav")